# Task 4: Open-Set Recognition — GPU Colab (account B, Drive-safe)

Use when **account A is out of GPU quota**.

1. Sign into Colab as the **GPU account** (account B).
2. Runtime → GPU.
3. This notebook clones GitHub into `/content/ATML-PA1`.
4. It also mounts **account B Drive** and syncs `task4/results/` after every stage so a disconnect does not lose checkpoints.
5. When finished, copy `task4_results_bundle.zip` from account-B Drive → account A → unzip into `task4/results/`.

**Hard rules:** CIFAR-10 val Acc for checkpoints; CIFAR-10 val only for thresholds; no CIFAR-100 in train.

In [ ]:
import torch

print("cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU — Runtime → Change runtime type → GPU, then re-run.")
print("device:", torch.cuda.get_device_name(0))

## Mount account-B Drive (backup target)

This is **not** account A’s code Drive — only used so disconnects don’t wipe results.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_BACKUP = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
print("backup dir:", DRIVE_BACKUP)

## Clone repo from GitHub

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/ttqureshi/ATML-PA1.git"
REPO_DIR = Path("/content/ATML-PA1")

if (REPO_DIR / ".git").is_dir():
    print("Repo exists — pulling latest main...")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "main"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"])
else:
    print("Cloning", REPO_URL)
    subprocess.check_call(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
!git log -1 --oneline

In [ ]:
%pip install -q -r requirements.txt

## Helper: sync results → Drive after each stage

In [ ]:
import shutil
from pathlib import Path

REPO_DIR = Path("/content/ATML-PA1")
RESULTS = REPO_DIR / "task4" / "results"
DRIVE_BACKUP = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")

def sync_to_drive(tag: str = ""):
    """Copy task4/results into Drive backup (overwrites same relative paths)."""
    DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
    dest = DRIVE_BACKUP / "results"
    if RESULTS.exists():
        shutil.copytree(RESULTS, dest, dirs_exist_ok=True)
    zip_path = DRIVE_BACKUP / "task4_results_bundle.zip"
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(DRIVE_BACKUP / "task4_results_bundle"), "zip", root_dir=RESULTS)
    print(f"[{tag}] synced → {dest}")
    print(f"[{tag}] zip → {zip_path} ({zip_path.stat().st_size} bytes)")

def restore_from_drive_if_any():
    """If a previous run left checkpoints on Drive, restore before training."""
    src = DRIVE_BACKUP / "results"
    if not src.exists():
        print("No prior Drive backup found.")
        return
    RESULTS.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, RESULTS, dirs_exist_ok=True)
    print("Restored prior results from Drive:")
    for p in sorted(RESULTS.rglob("*.pt")):
        print(" ", p, p.stat().st_size)

restore_from_drive_if_any()

## Run pipeline (background + Drive sync)

Skips a train stage if that checkpoint already exists (so you can resume after disconnect).

In [ ]:
import os, subprocess
from pathlib import Path

os.chdir("/content/ATML-PA1")
log = Path("/content/task4_pipeline.log")
done = Path("/content/task4_pipeline.done")
if done.exists():
    done.unlink()
log.write_text("starting\n")

# Resume-aware shell script: skip train if *_best.pt already present
script = r'''
set -e
cd /content/ATML-PA1
CKPT=task4/results/checkpoints
mkdir -p "$CKPT"

sync() {
  python - <<'PY'
from pathlib import Path
import shutil
RESULTS = Path("task4/results")
DEST = Path("/content/drive/MyDrive/ATML-PA1-task4-backup/results")
DEST.mkdir(parents=True, exist_ok=True)
if RESULTS.exists():
    shutil.copytree(RESULTS, DEST, dirs_exist_ok=True)
zp = Path("/content/drive/MyDrive/ATML-PA1-task4-backup/task4_results_bundle.zip")
if zp.exists():
    zp.unlink()
shutil.make_archive(str(zp.with_suffix("")), "zip", root_dir=RESULTS)
print("synced + zipped", zp, zp.stat().st_size)
PY
}

python -m task4.scripts.run_task4 --stages splits
sync

if [ ! -f "$CKPT/vanilla_best.pt" ]; then
  python -m task4.scripts.run_task4 --stages train_vanilla
  sync
else
  echo "SKIP train_vanilla (checkpoint exists)"
fi

if [ ! -f "$CKPT/gcsc_best.pt" ]; then
  python -m task4.scripts.run_task4 --stages train_gcsc
  sync
else
  echo "SKIP train_gcsc (checkpoint exists)"
fi

if [ ! -f "$CKPT/proser_best.pt" ]; then
  python -m task4.scripts.run_task4 --stages train_proser
  sync
else
  echo "SKIP train_proser (checkpoint exists)"
fi

python -m task4.scripts.run_task4 --stages extract_all
sync
python -m task4.scripts.run_task4 --stages eval
sync
touch /content/task4_pipeline.done
echo DONE
'''

proc = subprocess.Popen(
    ["bash", "-lc", script],
    stdout=open(log, "a"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("PID", proc.pid)
print("log: /content/task4_pipeline.log")
print("Drive backup: /content/drive/MyDrive/ATML-PA1-task4-backup/")
print("Keep this tab open; even if MCP disconnects, Drive sync should keep checkpoints.")

## Poll progress (re-run anytime)

In [ ]:
from pathlib import Path
import subprocess

log = Path("/content/task4_pipeline.log")
done = Path("/content/task4_pipeline.done")
print("done", done.exists())
r = subprocess.run(["bash", "-lc", "pgrep -af 'python3 -m task4' || true"], capture_output=True, text=True)
print("procs:\n", r.stdout)
if log.exists():
    lines = log.read_text(errors="ignore").splitlines()
    keep = [
        ln for ln in lines
        if ("epoch" in ln.lower() or ">>" in ln or "best" in ln or "SKIP" in ln
            or "synced" in ln or "DONE" in ln or "Error" in ln or "Traceback" in ln
            or "Wrote" in ln or "Evaluating" in ln)
        and "it/s" not in ln
    ]
    print("--- tail ---")
    print("\n".join(keep[-40:]))

bk = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")
print("Drive ckpts:")
for p in sorted((bk / "results" / "checkpoints").glob("*.pt")) if (bk / "results" / "checkpoints").exists() else []:
    print(" ", p.name, p.stat().st_size)
zp = bk / "task4_results_bundle.zip"
print("Drive zip exists:", zp.exists(), zp.stat().st_size if zp.exists() else "")

## After DONE: bring zip to account A

On account B Drive you should see `MyDrive/ATML-PA1-task4-backup/task4_results_bundle.zip`.

1. Download it (or share to account A).
2. On account A, unzip into `task4/results/`.
3. Tell the agent to verify locally.